<div style="background-color: #0F172A; padding: 20px; border-radius: 10px; border-left: 10px solid #C8F542; color: white;">
  <h1 style="color: #C8F542; margin: 0;">📌 ARKOS MI — Etapa 1: Expressões Regulares + spaCy</h1>
  <h3 style="color: #6366F1; margin-top: 5px;">Infraestrutura de Inteligência Corporativa | Módulo Marketing Intelligence (MI)</h3>
  <p style="font-size: 1.1em; color: #F4F2ED;">
    Nesta etapa, focamos na <b>Extração Estruturada</b> e na <b>Limpeza Progressiva</b> de dados não estruturados de startups (READMEs, Blogs, Redes Sociais).
  </p>
</div>

In [ ]:
import sys
import os
import re
import pandas as pd
import spacy

# Adicionando src ao path
sys.path.append(os.path.abspath('../src'))
from limpeza import *

# Inicializando spaCy para a Seção 2
try:
    nlp = spacy.load("pt_core_news_sm")
    print("✅ Módulo de Limpeza e spaCy PRONTOS!")
except:
    print("❌ Erro ao carregar spaCy. Instale com: python -m spacy download pt_core_news_sm")

# Texto da Startup Fintech "PayBR API v2.3.1"
readme_paybr = """
# PayBR API - Pagamentos Instantâneos v2.3.1

![Build](https://img.shields.io/travis/paybr/api.svg)

A @paybr_startup é o gateway definitivo para o Brasil. CPFs validos: 123.456.789-01 e 987.654.321-99. 
Fale conosco em contato@paybr.io ou acesse o Slack @atendimento_paybr.

## Changelog
### v2.3.1
- Fixes #201: Erro de timeout no Pix.
### v2.2.0
- Lançamento da carteira digital.

```bash
npm install paybr-sdk
```
Mais detalhes em https://paybr.io/docs/v2.3.1
"""

<h2 style="color: #C8F542; border-bottom: 2px solid #C8F542;">⚡ SEÇÃO 1 — Expressões Regulares (Regex)</h2>

As expressões regulares permitem que o motor MI da ARKOS identifique padrões complexos sem depender de modelos estatísticos lentos.

### 🔍 1.1 Extração com `re.findall()`

Utilizamos `re.findall()` para capturar todos os e-mails, versões, menções e referências estruturais (Trust Markers).

In [ ]:
# Extração de CPFs (técnica não incluída no src mas exigida no notebook)
padrao_cpf = r'\d{3}\.\d{3}\.\d{3}-\d{2}'
cpfs = re.findall(padrao_cpf, readme_paybr)

df_extracao = pd.DataFrame({
    "Tipo": ["E-mails", "Versões", "Menções (@)", "Issues (#)", "CPFs"],
    "Resultados": [
        extrair_emails(readme_paybr),
        extrair_versoes(readme_paybr),
        extrair_mentions(readme_paybr),
        extrair_issue_refs(readme_paybr),
        cpfs
    ]
})
df_extracao

### 🧹 1.2 Limpeza Progressiva com `re.sub()`

Remação de ruído (Markdown, blocos de código e URLs) para focar no conteúdo semântico.

In [ ]:
texto_limpo_markdown = limpar_markdown(readme_paybr)

print("TABELA DE LIMPEZA PROGRESSIVA")
print("-" * 80)
print(f"| ANTES (Markdown Bruto) | {readme_paybr[:50]}... |")
print(f"| DEPOIS (Texto Limpo)   | {texto_limpo_markdown[:50]}... |")
print("-" * 80)

### 📊 1.3 Análise de Qualidade com `re.search()`

Verificamos se o README contém seções vitais seguindo o Framework ARKOS MI.

In [ ]:
score_qualidade = verificar_qualidade_readme(readme_paybr)
pd.DataFrame(list(score_qualidade.items()), columns=["Critério", "Presente?"])

### 🧩 1.4 Divisão de Changelog com `re.split()`

Segmentamos o texto por headers de versão para análise temporal de evolução da startup.

In [ ]:
blocos_changelog = split_changelog_por_versao(readme_paybr)
dict_changelog = {f"Versão {i+1}": bloco[:100] for i, bloco in enumerate(blocos_changelog)}
dict_changelog

### 🛡️ 1.5 Moderação com `re.sub(re.IGNORECASE)`

Garantindo a reputação da marca através de filtros de sensibilidade linguística.

In [ ]:
texto_ofensivo = "Essa API é uma porcaria e o serviço é muito LENTO!"
lista_black = ["porcaria", "lento"]
texto_moderado = moderar_conteudo(texto_ofensivo, lista_black)

print(f"Original: {texto_ofensivo}")
print(f"Moderado: {texto_moderado}")

<h2 style="color: #C8F542; border-bottom: 2px solid #C8F542;">🧩 SEÇÃO 2 — spaCy Básico</h2>

A transição da extração por padrões (Regex) para o entendimento morfológico e normalizado.

### 🪙 2.1 Tokenização

O primeiro passo do spaCy: decompor o texto limpo em unidades de significado (tokens).

In [ ]:
df_tokens = tokenizar(texto_limpo_markdown, nlp)
df_tokens.head(15)

### 🌱 2.2 Lematização e Normalização

Reduzindo a variação morfológica (lematização) e preparando o texto para modelos estatísticos (normalização).

In [ ]:
# Normalização completa via ARKOS Pipeline
texto_normalizado = limpar_para_spacy(readme_paybr)

# Comparativo de Lematização
dados_lemas = normalizar_e_lematizar(readme_paybr, nlp)
pd.DataFrame({"Original": corpus_startups[0].split()[:5], "Lema": dados_lemas["tokens_lematizados"][:5]})
print(f"\n🔥 Texto Normalizado ARKOS: {texto_normalizado[:150]}...")

<div style="background-color: #1F242D; padding: 15px; border-radius: 5px; color: white;">
  <h3 style="color: #6366F1;">💡 Conclusão: Regex vs spaCy (Aula 2)</h3>
  
  <b>Quando usar Regex (re)?</b>
  <ul>
    <li>Padrões fixos (e-mails, CPFs, URLs).</li>
    <li>Limpeza rápida e substituições globais (re.sub).</li>
    <li>Divisão de grandes documentos (re.split).</li>
    <li><i>Limitação:</i> Não entende contexto ou gramática.</li>
  </ul>
  
  <b>Quando usar spaCy?</b>
  <ul>
    <li>Análise sintática e morfológica (POS Tagging).</li>
    <li>Entendimento de entidades (NER).</li>
    <li>Normalização profunda (Lematização).</li>
    <li>Vetorização de palavras.</li>
    <li><i>Limitação:</i> Mais lento e "pesado" que Regex.</li>
  </ul>
</div>